# MLOPs Lab 6: Experiment Tracking and Model Registry with MLflow

## 1. Aim

To record, compare and manage machine learning experiments systematically using MLflow, and to store the best-performing model in a model registry so that it can be identified and reused reliably.

## 2. Theory

### 2.1 The problem

A model is usually trained many times with different settings. If results are only printed in the notebook, they are lost when the kernel restarts, and it becomes difficult to say which settings gave the best result. **Experiment tracking** solves this by saving every run automatically.

### 2.2 MLflow

MLflow is an open-source tool for tracking machine learning experiments. For every training run it records:

- **Parameters**: the settings chosen, for example `max_depth = 4`
- **Metrics**: the results obtained, for example `mae_minutes = 3.31`
- **Artifacts**: files produced by the run, mainly the trained model
- **Tags**: labels added by the user, for example `sweep = t3`

The **Model Registry** stores a model under a name and a version number, for example *delivery-time-model, version 1*.

### 2.3 Where MLflow stores its records

Older tutorials store runs in a folder named `./mlruns`. In MLflow 3 this is no longer recommended, so we use a SQLite database file, `mlflow.db`, which is created automatically in the notebook's folder.

**Metric used:** Mean Absolute Error (MAE), in minutes. A lower MAE means a better model.

## 3. Procedure Overview

1. Import libraries and check the MLflow installation
2. Prepare the dataset
3. Observe the problem: training without tracking
4. Connect MLflow to a database
5. Record the first tracked run
6. Read the recorded run back
7. Record a sweep of several settings
8. Find the best run by querying the database
9. Save the best model and register it
10. Inspect the Model Registry
11. Load the registered model and make a prediction
12. Explore the runs in the MLflow web interface
13. Complete Task: Breast Cancer Diagnosis

**Installation note.** If MLflow is not yet installed, run the following command once in a terminal (an internet connection with access to `pypi.org` is required):

```
pip install mlflow
```

## Step 1: Import Libraries and Check the Installation

In [1]:
# Data handling
import numpy as np
import pandas as pd

# Model building and evaluation (scikit-learn)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Experiment tracking
import mlflow
from mlflow import MlflowClient

print("MLflow version:", mlflow.__version__)

MLflow version: 3.17.1


## Step 2: Prepare the Dataset

In [2]:
# Fix the random seed so that everyone obtains the same data
np.random.seed(42)

number_of_orders = 500

# Create the input features
df = pd.DataFrame()
df["distance_km"] = np.random.uniform(1, 15, number_of_orders)
df["items"]       = np.random.randint(1, 10, number_of_orders)
df["hour"]        = np.random.randint(8, 23, number_of_orders)

# Create the target: base time + time for distance + time for items + noise
noise = np.random.normal(0, 3, number_of_orders)
df["delivery_minutes"] = 10 + 3 * df["distance_km"] + 1.5 * df["items"] + noise

# Separate the inputs (X) from the target (y)
X = df[["distance_km", "items", "hour"]]
y = df["delivery_minutes"]

# Split into 80% training data and 20% test data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", len(X_train))
print("Test rows    :", len(X_test))
df.head()

Training rows: 400
Test rows    : 100


,distance_km,items,hour,delivery_minutes
0,6.243562,6,18,36.064045
1,14.310000,1,21,55.905380
2,11.247915,9,14,58.715004
3,9.381219,1,8,37.832316
4,3.184261,5,9,26.350811


## Step 3: Observe the Problem (Training Without Tracking)

We train three decision trees with different values of `max_depth` and print their MAE. Nothing is saved.

**Discussion point.** If the kernel is restarted now, these numbers are lost. After a week, it would be difficult to say which depth gave the best result. This is the problem that experiment tracking addresses.

In [3]:
for depth in [2, 5, 10]:
    # Train a decision tree with the chosen depth
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)

    # Evaluate it on the test set
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)

    print("max_depth =", depth, "   MAE =", round(mae, 2))

max_depth = 2    MAE = 4.59
max_depth = 5    MAE = 3.31
max_depth = 10    MAE = 4.07


## Step 4: Connect MLflow to a Database

Two settings are required before any run can be recorded:

1. **Tracking URI**: the location where MLflow stores its records. `sqlite:///mlflow.db` tells MLflow to use a SQLite database file named `mlflow.db` in the current folder. The file is created automatically if it does not exist.
2. **Experiment name**: the group to which our runs will belong. If the experiment does not exist, MLflow creates it.

In [4]:
# Store all MLflow records in a local SQLite database file
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# All runs in this notebook will belong to this experiment
mlflow.set_experiment("P06-delivery")

print("Tracking URI:", mlflow.get_tracking_uri())

2026/10/27 11:28:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/27 11:28:46 INFO mlflow.store.db.utils: Updating database tables
2026/10/27 11:28:48 INFO mlflow.tracking.fluent: Experiment with name 'P06-delivery' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db


## Step 5: Record the First Tracked Run

A run is started with `mlflow.start_run()`. Everything logged inside the `with` block is stored as part of that run. We use three logging functions:

- **`mlflow.log_param(name, value)`**: An input setting (parameter)
- **`mlflow.log_metric(name, value)`**: A numerical result (metric)
- **`mlflow.set_tag(name, value)`**: A descriptive label (tag)

Compared with Step 3, only these logging lines are new; the training code is unchanged.

In [5]:
with mlflow.start_run(run_name="first-tree"):

    depth = 4

    # Train and evaluate the model (same as before)
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))

    # Record the parameters (the settings we chose)
    mlflow.log_param("max_depth", depth)
    mlflow.log_param("test_size", 0.2)

    # Record the metric (the result we obtained)
    mlflow.log_metric("mae_minutes", mae)

    # Record a tag (a label of our choice)
    mlflow.set_tag("student", "demo")

print("Run recorded. MAE =", round(mae, 2))

Run recorded. MAE = 3.47


## Step 6: Read the Recorded Run Back

`mlflow.search_runs()` reads runs from the database and returns them as a pandas DataFrame. The column names follow a fixed pattern:

- **`params.`**: Parameters, for example `params.max_depth`
- **`metrics.`**: Metrics, for example `metrics.mae_minutes`
- **`tags.`**: Tags, for example `tags.mlflow.runName` (the run name)

The result is read from the database, not from the notebook's memory, so it remains available even after the kernel is restarted.

In [6]:
# Read all runs of the experiment from the database
runs = mlflow.search_runs(experiment_names=["P06-delivery"])

# Display only the columns we are interested in
runs[["tags.mlflow.runName", "params.max_depth", "metrics.mae_minutes"]]

,tags.mlflow.runName,params.max_depth,metrics.mae_minutes
0,first-tree,4,3.465814


## Step 7: Record a Sweep of Several Settings

A **sweep** means training the same model several times with different values of a parameter. Here we try six values of `max_depth`. Each iteration of the loop creates a separate run in MLflow.

In [7]:
depth_values = [1, 2, 3, 5, 8, 12]

for depth in depth_values:

    # Each loop iteration is recorded as a separate run
    with mlflow.start_run(run_name="tree-depth-" + str(depth)):

        model = DecisionTreeRegressor(max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        mae = mean_absolute_error(y_test, model.predict(X_test))

        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("mae_minutes", mae)

    print("max_depth =", depth, "   MAE =", round(mae, 2))

max_depth = 1    MAE = 6.11
max_depth = 2    MAE = 4.59
max_depth = 3    MAE = 4.04
max_depth = 5    MAE = 3.31
max_depth = 8    MAE = 3.96
max_depth = 12    MAE = 4.39


**Observation.** Test MAE decreases as depth increases up to max_depth = 5 (MAE = 3.31), and then increases again for deeper trees. This pattern is consistent with underfitting at small depths (the tree is too simple) and overfitting at large depths (the tree fits noise in the training data). Among the values tried, max_depth = 5 gives the lowest test error.

## Step 8: Find the Best Run by Querying the Database

Instead of reading earlier output, we ask MLflow to sort all runs by MAE in ascending order (smallest first). The first row is therefore the best run.

- `filter_string="attributes.status = 'FINISHED'"` keeps only runs that completed successfully, ignoring any run that failed with an error.
- `order_by=["metrics.mae_minutes ASC"]` sorts by MAE, lowest first.
- `max_results=1` returns only the top run.

In [8]:
best = mlflow.search_runs(
    experiment_names=["P06-delivery"],
    filter_string="attributes.status = 'FINISHED'",   # only successful runs
    order_by=["metrics.mae_minutes ASC"],              # lowest MAE first
    max_results=1,                                     # only the best run
)

# The first (and only) row is the best run
best_run_name = best["tags.mlflow.runName"][0]
best_depth    = best["params.max_depth"][0]
best_mae      = best["metrics.mae_minutes"][0]

print("Best run  :", best_run_name)
print("Best depth:", best_depth)
print("Best MAE  :", round(best_mae, 2))

Best run  : tree-depth-5
Best depth: 5
Best MAE  : 3.31


**Note.** MLflow stores parameters as text (strings). Therefore `best_depth` is the text `"5"`, not the number `5`. It must be converted with `int()` before it is passed to a model, as done in the next step.

## Step 9: Save the Best Model and Register It

We retrain the model with the best depth and save the trained model as an **artifact** using `mlflow.sklearn.log_model()`. The arguments are:

- **`sk_model`**: The trained scikit-learn model to be saved
- **`name`**: Name of the folder, inside the run, in which the model is stored
- **`registered_model_name`**: Registers the model in the Model Registry under this name
- **`serialization_format`**: The file format used to save the model. We use `"cloudpickle"`, the standard Python format, because it works reliably on all lab machines.

The first time this cell is executed, MLflow creates **version 1** of `delivery-time-model`. Executing it again creates version 2, and so on.

In [9]:
# Parameters are stored as text, so convert the depth to an integer
best_depth = int(best_depth)

with mlflow.start_run(run_name="winner-saved"):

    # Retrain the model using the best depth
    model = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))

    mlflow.log_param("max_depth", best_depth)
    mlflow.log_metric("mae_minutes", mae)

    # Save the trained model as an artifact and register it
    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        registered_model_name="delivery-time-model",
        serialization_format="cloudpickle",
    )

print("Model saved and registered. MAE =", round(mae, 2))

2026/10/27 11:28:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model saved and registered. MAE = 3.31


Successfully registered model 'delivery-time-model'.
Created version '1' of model 'delivery-time-model'.


## Step 10: Inspect the Model Registry

`MlflowClient` gives lower-level access to MLflow. Here we use it to list every registered version of `delivery-time-model`, together with the ID of the run that produced it. The run ID links each model version back to the exact parameters and metrics that created it.

In [10]:
client = MlflowClient()

versions = client.search_model_versions("name='delivery-time-model'")

for v in versions:
    print("Version:", v.version, "   Created by run:", v.run_id)

Version: 1    Created by run: 8ef5e51d31284e168809c1618aa98416


## Step 11: Load the Registered Model and Make a Prediction

A registered model is loaded using a URI of the form:

```
models:/<model name>/<version number>
```

Any team member with access to the same MLflow store can load exactly the same model with this one line, without needing to locate or exchange the model file manually.

In [11]:
# Load version 1 of the registered model
loaded_model = mlflow.sklearn.load_model("models:/delivery-time-model/1")

# A new order: 8 km away, 3 items, placed at 7 pm (hour 19)
new_order = pd.DataFrame({
    "distance_km": [8],
    "items":       [3],
    "hour":        [19],
})

predicted_time = loaded_model.predict(new_order)[0]
print("Predicted delivery time:", round(predicted_time, 1), "minutes")

Predicted delivery time: 42.4 minutes


## Step 12: Explore the Runs in the MLflow Web Interface

MLflow includes a web interface for browsing runs. The cell below starts it in the background, so that the notebook remains usable. It works on Windows, macOS and Linux.

**Run this cell only once.** Running it a second time will fail because port 5001 is already in use.

After about 10 seconds, open **http://127.0.0.1:5001** in a web browser.

> **Why port 5001?** On macOS, port 5000 is used by the AirPlay Receiver service, so the default port often fails. Port 5001 works on all systems.

**Alternative.** The interface can also be started from a terminal, opened in the same folder as this notebook:

```
python -m mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5001
```

> **Windows users:** open **Anaconda Prompt** (not the plain Command Prompt), move to the notebook's folder using `cd`, and then run the command above. If a Windows Defender Firewall prompt appears, click **Allow**.

In [12]:
import subprocess
import sys

# Start the MLflow web interface as a background process
ui_process = subprocess.Popen([
    sys.executable, "-m", "mlflow", "ui",
    "--backend-store-uri", "sqlite:///mlflow.db",
    "--port", "5001",
])

print("MLflow UI is starting. Open http://127.0.0.1:5001 in your browser.")

MLflow UI is starting. Open http://127.0.0.1:5001 in your browser.


### Using the web interface

1. The interface may open in **GenAI** mode, which shows LLM traces and will appear empty. Click **Model training** at the top left.
2. Open the experiment **P06-delivery** and select **Runs** (or **Training runs**).
3. Click **Columns** and add the metric `mae_minutes` and the parameter `max_depth`. Click the `mae_minutes` column heading to sort the runs.
4. Select three or four `tree-depth-*` runs using the check boxes and click **Compare** to view a chart of `max_depth` against `mae_minutes`.
5. Open the run `winner-saved` and view its **Artifacts**. The `model` folder contains the saved model and the file `MLmodel`, which describes it.
6. Open **Models** (or **Model registry**) in the left menu to see `delivery-time-model` and its versions.

To stop the web interface at the end of the session, run the following cell.

In [13]:
# Stop the MLflow web interface (run this only at the end)
#ui_process.terminate()

# Task: Experiment Tracking for Breast Cancer Diagnosis

Use the **Breast Cancer Wisconsin dataset**, which is built into scikit-learn (load it with `from sklearn.datasets import load_breast_cancer`, so no download is needed), to build a model that classifies a tumour as **malignant** or **benign**, and track all your experiments with MLflow. Split the data into 80% training and 20% test sets, connect MLflow to `sqlite:///mlflow.db`, and create an experiment named `cancer-diagnosis`. First train a **Logistic Regression** baseline in a run named `logreg-baseline`, then train a **Random Forest** for every combination of `n_estimators` (10, 50, 100) and `max_depth` (3, 5, 10), giving nine runs tagged `sweep = rf`. For every run, log the parameters, the **accuracy** and the **F1-score**, and save the model. Then use `mlflow.search_runs()` to find the best run by F1-score without reading earlier output, register that model as `cancer-classifier`, load it back using `models:/cancer-classifier/1`, and predict the first five test samples. Finally, take a screenshot of the MLflow run table showing all runs with their accuracy and F1-score, and write two or three sentences on which model performed best and whether the Random Forest improved on the baseline.

In [14]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("cancer-diagnosis")

2026/10/27 11:29:08 INFO mlflow.tracking.fluent: Experiment with name 'cancer-diagnosis' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/btash/AppData/Local/Packages/5319275A.WhatsAppDesktop_cv1g1gvanyjgm/LocalState/sessions/DC3B03865C085E0BA1989B019B7519758D863D73/transfers/2026-43/mlruns/2', creation_time=1793080748885, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1793080748885, lifecycle_stage='active', name='cancer-diagnosis', tags={}, trace_location=None, workspace='default'>

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

with mlflow.start_run(run_name="logreg-baseline"):
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    mlflow.log_param("max_iter", 1000)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(sk_model=model, name="model", serialization_format="cloudpickle")

    print("Baseline Accuracy:", acc)
    print("Baseline F1-score:", f1)

c:\Users\btash\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/10/27 11:29:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Baseline Accuracy: 0.956140350877193
Baseline F1-score: 0.9655172413793104


In [17]:
from sklearn.ensemble import RandomForestClassifier

n_estimators_list = [10, 50, 100]
max_depth_list = [3, 5, 10]

for n in n_estimators_list:
    for d in max_depth_list:
        with mlflow.start_run(run_name="rf-n" + str(n) + "-d" + str(d)):
            mlflow.set_tag("sweep", "rf")
            model = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            acc = accuracy_score(y_test, preds)
            f1 = f1_score(y_test, preds)

            mlflow.log_param("n_estimators", n)
            mlflow.log_param("max_depth", d)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)

            mlflow.sklearn.log_model(sk_model=model, name="model", serialization_format="cloudpickle")
            print("n_estimators =", n, "max_depth =", d, "Accuracy =", round(acc, 4), "F1 =", round(f1, 4))

2026/10/27 11:29:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/27 11:29:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators = 10 max_depth = 3 Accuracy = 0.9561 F1 = 0.965


2026/10/27 11:29:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators = 10 max_depth = 5 Accuracy = 0.9561 F1 = 0.965
n_estimators = 10 max_depth = 10 Accuracy = 0.9561 F1 = 0.965


2026/10/27 11:29:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

n_estimators = 100 max_depth = 3 Accuracy = 0.9649 F1 = 0.9722


2026/10/27 11:30:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators = 100 max_depth = 5 Accuracy = 0.9649 F1 = 0.9722


2026/10/27 11:30:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators = 100 max_depth = 10 Accuracy = 0.9649 F1 = 0.9722


In [18]:
best = mlflow.search_runs(
    experiment_names=["cancer-diagnosis"],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.f1_score DESC"],
    max_results=1
)

best_run_id = best["run_id"][0]
best_run_name = best["tags.mlflow.runName"][0]
best_f1 = best["metrics.f1_score"][0]
best_acc = best["metrics.accuracy"][0]

print("Best Run ID:", best_run_id)
print("Best Run Name:", best_run_name)
print("Best F1-Score:", best_f1)
print("Best Accuracy:", best_acc)

Best Run ID: d11af4ce6d144066a0d7b4f5e0e366b6
Best Run Name: rf-n100-d10
Best F1-Score: 0.9722222222222222
Best Accuracy: 0.9649122807017544


In [19]:
model_uri = "runs:/" + best_run_id + "/model"
mlflow.register_model(model_uri, "cancer-classifier")

Successfully registered model 'cancer-classifier'.
2026/10/27 11:30:22 WARNING mlflow.tracking._model_registry.fluent: Run with id d11af4ce6d144066a0d7b4f5e0e366b6 has no artifacts at artifact path 'model', registering model based on models:/m-2044ac4609a64bc4a2d10ee06f3954c3 instead
Created version '1' of model 'cancer-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1793080822535, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1793080822535, metrics=None, model_id=None, name='cancer-classifier', params=None, run_id='d11af4ce6d144066a0d7b4f5e0e366b6', run_link=None, source='models:/m-2044ac4609a64bc4a2d10ee06f3954c3', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [20]:
loaded_model = mlflow.sklearn.load_model("models:/cancer-classifier/1")
predictions = loaded_model.predict(X_test[:5])
print("Predictions:", predictions)
print("Actual:", y_test[:5])

Predictions: [1 0 0 1 1]
Actual: [1 0 0 1 1]


### Conclusion

The Random Forest model with 50 estimators and a max depth of 3 performed best, achieving an F1-score of 0.9722 and an accuracy of 0.9649. In comparison, the Logistic Regression baseline achieved an F1-score of 0.9655 and an accuracy of 0.9561. Thus, the Random Forest model improved upon the baseline model across both evaluation metrics.